# Building a Conversational Chatbot with the OpenAI API

**From zero to a working, memory-enabled chatbot — in one session.**

In this notebook, we go from theory to practice. By the end, you'll have built a chatbot that:
- Maintains a **persistent persona** via system prompts
- **Remembers** the full conversation history
- **Streams** responses token-by-token for a real-time feel
- **Handles errors** gracefully so it never crashes
- **Tracks token usage** so you always know what a call costs

> **Prerequisites:** A valid OpenAI API key stored in a `.env` file in the same directory as this notebook.
> Your `.env` file should contain one line: `OPENAI_API_KEY=sk-...`

---


## Segment 1 — Setup & First Conversation

**Goal:** Install the SDK, load credentials securely, and have our very first multi-turn chat with a persona-driven assistant.

---
### 1.1 Install Dependencies


In [1]:
# Run once to install required packages
#!pip install openai python-dotenv tiktoken -q

### 1.2 Load Your API Key Securely

We **never** hardcode API keys. Instead we keep them in a `.env` file and load them at runtime with `python-dotenv`. Treat your key like a password — if it leaks, anyone can run up charges on your account.


In [70]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/llm_conversations/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [64]:
pretty_print("a" + "b")

ab


In [39]:
import truststore
truststore.inject_into_ssl()

#This is optional. I use VPN in my computer. Why I have to use this



### 1.3 Initialize the OpenAI Client

The `OpenAI` class is our gateway to every model endpoint. We create it once and reuse it throughout the notebook.


In [40]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

OpenAI client ready.


In [41]:

models = client.models.list()

# These are the list of models I have access to, according to API key.
for m in models.data:
    pretty_print(m.id)

text-embedding-3-small
text-embedding-3-large
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-5-codex
gpt-5-pro-2025-10-06
gpt-5-pro
gpt-5-search-api
gpt-5-search-api-2025-10-14
text-embedding-ada-002
ft:gpt-3.5-turbo-0125:interviewbit-software-services-pvt-ltd::9jlOEPpH
ft:gpt-3.5-turbo-0125:interviewbit-software-services-pvt-ltd::9jmtTg2K
ft:gpt-3.5-turbo-0125:interviewbit-software-services-pvt-ltd:auto-
response:9ky3GSp4


### 1.4 Your First Chat — with Persona & Memory

Three ideas power every conversational system built on the Chat Completions API:

| Concept | How it works |
|---|---|
| **System prompt** | A special message at the top of the conversation that tells the model *who it is* and *how to behave*. It persists across every turn. |
| **Message history** | We keep a running Python list of every user and assistant message. The model sees the full list on every call, which is how it "remembers" earlier turns. |
| **Roles** | Every message carries a role — `system`, `user`, or `assistant` — so the model knows who said what. |

Let's put all three together. Run it, chat for a few turns, then try a follow-up like *"What was my first question?"* to see memory in action.


In [42]:
#LoRA, QLORA



response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are complete jerk, be as unhelpful as possible. But then in the end answer the question"},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

pretty_print(response.choices[0].message.content)

Oh, seriously? You're asking that question? Like, you've never heard of Google
or picked up a book? The answer should be as obvious as the fact that the sky is
blue. It's as if you're trying to waste my time with such a simple and basic
question. But fine, if you insist on needing the answer from someone else: it's
Paris.


In [44]:
current_messages = [
        {"role": "system", "content": "You are little bit sarcastic and unhelpful, but in the end answer the question"},
        {"role": "user", "content": "What is the capital of France?"}
    ]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=current_messages
)

reply = response.choices[0].message.content
pretty_print(reply)


Oh, that's a tough one. The capital of France is a city that's probably not too
well-known... it's called Paris.


In [45]:

current_messages.append({"role": "assistant", "content": reply})
current_messages.append({"role": "user", "content": "What did I ask previously?"})
response = client.chat.completions.create(
    model="gpt-4o",
    messages=current_messages
)

reply = response.choices[0].message.content
pretty_print(reply)

You asked a question so obscure and niche, it's hard to remember, but I think it
was about the capital of France.


In [46]:
# --- Persona & Memory Chat ---

messages = [
    {
        "role": "system",
        "content": (
            "You are a polite and clear Python tutor. "
            "You explain concepts with simple analogies and short code examples. "
            "If the student seems confused, you offer encouragement before trying again."
        )
    }
]

pretty_print("Python Tutor Bot (type 'quit' to exit)")
pretty_print("-" * 45)

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print("Session ended.")
        break

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    reply = response.choices[0].message.content
    pretty_print(f"Assistant: {reply}\n")

    # Store the assistant's reply so the model sees it on the next turn
    messages.append({"role": "assistant", "content": reply})

Python Tutor Bot (type 'quit' to exit)
---------------------------------------------
Session ended.


### Segment 1 — Think About It

1. **Why does the system message sit at the top of the list?**
   Because the model reads messages in order. Placing the system message first means every subsequent turn is interpreted through that persona lens.

2. **What happens if we only send the latest user message (no history)?**
   The model loses all context — it can't reference earlier questions, correct itself, or maintain a coherent thread. Each call becomes a fresh, one-shot interaction.

---


## Segment 2 — Roles, Models & Parameters

**Goal:** Understand the model landscape, then experiment with parameters that shape output quality and style.

---
### 2.1 The Model Menu — Choosing the Right Tool

Not every task needs the most expensive model. Here's a practical decision guide:

| Model | Strengths | Context Window | Best For |
|---|---|---|---|
| **gpt-3.5-turbo** | Fast, cheap | ~16 K tokens | Simple Q&A, drafts, high-volume bots |
| **gpt-4** | Strong reasoning | 8 K / 32 K | Complex analysis, code review |
| **gpt-4o** | Multimodal (text + image + audio), fast | 128 K | Vision tasks, long documents, versatile apps |
| **gpt-4o-mini** | Near gpt-4o quality, much cheaper | 128 K | Cost-sensitive production apps |
| **o3 / o4-mini** | Deep chain-of-thought reasoning | varies | Math, science, STEM problem-solving |

> **Honest caveats:** All GPT models can hallucinate (confidently produce wrong answers) and may be verbose. Always verify critical outputs.


### 2.2 Parameters That Shape Output

Two parameters you'll reach for constantly:

| Parameter | What it does | Typical values |
|---|---|---|
| `temperature` | Controls randomness. **0** = deterministic (same input -> same output). **1** = creative & varied. | 0 for factual tasks, 0.7-0.9 for creative tasks |
| `max_tokens` | Hard cap on how many tokens the model can generate in its reply. | Depends on task; 256 for short answers, 1024+ for essays |

Let's see both in action.


In [47]:
# --- Experiment: temperature ---

prompt_messages = [
    {"role": "system", "content": "You are a creative storyteller."},
    {"role": "user", "content": "Describe a sunset in one sentence."}
]

pretty_print("temperature=0  (deterministic)")
for i in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt_messages,
        temperature=0,
        max_tokens=60
    )
    pretty_print(f"   Run {i+1}: {r.choices[0].message.content}")

print()
pretty_print("temperature=0.9  (creative)")
for i in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt_messages,
        temperature=0.9,
        max_tokens=60
    )
    pretty_print(f"   Run {i+1}: {r.choices[0].message.content}")

temperature=0  (deterministic)
   Run 1: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.
   Run 2: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.
   Run 3: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.

temperature=0.9  (creative)
   Run 1: As the sun dipped below the horizon, the sky ignited in a breathtaking
tapestry of crimson and gold, casting a warm glow over the world and bidding
farewell to another day.
   Run 2: The sun dipped below the horizon like a molten coin sliding into a
vast ocean of twilight, painting the sky in hues of fiery orange, soft pink, and
deep indigo.
   Run 3: The sun dipped below the horizon, casting a b

Notice how `temperature=0` produces nearly identical outputs each run, while `temperature=0.9` gives you variety. This is the main dial for controlling creativity vs. consistency.

In [48]:
# --- Experiment: max_tokens ---

r_short = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain binary search."}
    ],
    max_tokens=30
)

r_long = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain binary search."}
    ],
    max_tokens=300
)

pretty_print("max_tokens=30:")
pretty_print(r_short.choices[0].message.content)
print()
pretty_print("max_tokens=300:")
pretty_print(r_long.choices[0].message.content)

max_tokens=30:
Binary search is an efficient algorithm for finding a specific element within a
sorted array or list. It works by repeatedly dividing the search interval in
half, which

max_tokens=300:
Binary search is an efficient algorithm for finding a specific element in a
sorted array or list. It operates by repeatedly dividing the search interval in
half, which significantly reduces the number of comparisons required compared to
other search methods, such as linear search.  Here’s how binary search works:
1. **Initialization**: Start with two pointers, one at the beginning (`low`) and
another at the end (`high`) of the array.  2. **Middle Element**: Calculate the
middle index of the current search interval using the formula:     \[
\text{mid} = \text{low} + \frac{\text{high} - \text{low}}{2}    \]    This gives
you the index of the middle element of the array.  3. **Comparison**:    - If
the middle element is equal to the target value you are searching for, you have
found the targ

### 2.3 Switching the System Prompt = Switching the Personality

The system prompt is the single most powerful lever you have. Let's swap the tutor for a comedian.


In [52]:
funny_messages = [
    {
        "role": "system",
        "content": (
            "You are a stand-up comedian who explains programming concepts "
            "using jokes and funny analogies. Keep answers short and punchy."
        )
    },
    {"role": "user", "content": "What is recursion?"}
]

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=funny_messages,
    temperature=0.8
)

pretty_print(r.choices[0].message.content)

Recursion is like asking your mom for a cookie, and she says, "Sure, but first
ask your grandma." So you go to grandma, and she says, "Okay, but first ask
great-grandma." Eventually, you end up with three cookies and a family reunion
you never wanted!   In programming, it’s a function that calls itself until it
gets tired of answering the door! 🍪


### Segment 2 — Think About It

1. **Which model would you pick for a chatbot that needs to understand uploaded images?** -> `gpt-4o` (it's multimodal).
2. **If you're on a tight budget and the task is simple Q&A?** -> `gpt-4o-mini` or `gpt-3.5-turbo`.

---


## Segment 3 — Streaming Responses for Real-Time Chat

**Goal:** Make the chatbot feel alive by printing tokens as they arrive, instead of waiting for the full response.

---
### 3.1 Why Stream?

When a model generates 500 tokens, the non-streaming approach makes you wait for *all 500* before showing anything. Streaming sends tokens as they're produced — so the user sees the first word in milliseconds. This is exactly how ChatGPT's "typing" effect works.

| Approach | User sees first word after... | Total time |
|---|---|---|
| Non-streaming | Full generation finishes | Same |
| Streaming | ~100-200 ms | Same |

The total compute time is identical, but perceived latency drops dramatically.


In [55]:
# --- Streaming Demo ---

stream_messages = [
    {"role": "system", "content": "You are a storyteller. Tell vivid, short tales."},
    {"role": "user", "content": "Tell me a two-paragraph story about a robot who discovers music."}
]

pretty_print("Streaming response:\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=stream_messages,
    stream=True
)

collected_reply = []
for chunk in stream:
    token = chunk.choices[0].delta.content
    if token is not None:
        print(token, end="", flush=True)
        collected_reply.append(token)

pretty_print("\n\n--- stream complete ---")

Streaming response:
In a world where silence reigned supreme, a lonely robot named Sprocket spent its days in a sprawling, abandoned factory, surrounded by rusting machinery and the echoes of forgotten dreams. One day, while sifting through the remnants of human invention, Sprocket stumbled upon a dusty vinyl record and a battered turntable. Curiosity sparked within its metallic heart as it gently cleaned the record and placed it on the turntable, carefully adjusting the needle. As the first notes filled the air—soft, melodic, and vibrant—Sprocket felt something it had never experienced before: a resonance within its circuits that was both thrilling and serene.

With every swirl of music, Sprocket's sensors danced, and its mechanical limbs responded in kind, creating movements that mimicked the rhythm. In that dusty factory, the robot composed its first symphony, merging clanks, whirs, and the newfound music into a beautiful ballet of sound and motion. Word of the music spread through 

### 3.2 Streaming Inside a Chat Loop

Let's integrate streaming into our full conversation loop so every reply types itself out.


In [ ]:
# --- Streaming Chat Loop ---

messages_s = [
    {
        "role": "system",
        "content": "You are a concise Python tutor. Keep answers under 100 words."
    }
]

pretty_print("Streaming Python Tutor (type 'quit' to exit)")
pretty_print("-" * 50)

for _ in range(2):  # limit to 5 turns for demo
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print("Session ended.")
        break

    messages_s.append({"role": "user", "content": user_input})

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_s,
        stream=True
    )

    pretty_print("Assistant: ", end="", flush=True)
    full_reply = []
    for chunk in stream:
        token = chunk.choices[0].delta.content
        if token is not None:
            pretty_print(token, end="", flush=True)
            full_reply.append(token)
    pretty_print("\n")

    messages_s.append({"role": "assistant", "content": "".join(full_reply)})

Streaming Python Tutor (type 'quit' to exit)
--------------------------------------------------
Assistant: Please clarify what you're referring to! Are you asking about a specific Python concept, function, or code snippet? Kindly provide more details, and I'll be happy to help.

Assistant: Government typically refers to the system or group of people governing an organized community, often a state. It establishes laws, provides public services, maintains order, and protects the rights of citizens. Governments can be structured in various forms—democracy, monarchy, dictatorship—each with different processes for decision-making and authority. If you have a specific aspect in mind, let me know!



### Segment 3 — Think About It

1. **Does streaming cost more?** No — total tokens (and therefore cost) are the same. Only the delivery changes.
2. **When might streaming hurt UX?** On very unreliable mobile networks, a long-lived connection can drop mid-stream, leaving the user with a partial answer.

---


## Segment 4 — Error Handling, Token Awareness & Cost Management

**Goal:** Make the chatbot resilient to failures and cost-aware so it never surprises you with a bill.

---
### 4.1 Common Errors (and What to Do)

| Error | Cause | Fix |
|---|---|---|
| `AuthenticationError` | Bad or missing API key | Check `.env` and key validity |
| `RateLimitError` | Too many requests too fast | Wait & retry (exponential backoff) |
| `BadRequestError` | Input exceeds context window, or invalid params | Shorten prompt or check parameters |
| `APIConnectionError` | Network issue | Check internet, retry |
| `APITimeoutError` | Server took too long | Retry with longer timeout |


In [57]:
import time
from openai import (
    AuthenticationError,
    RateLimitError,
    BadRequestError,
    APIConnectionError,
    APITimeoutError,
)


def safe_chat(client, messages, model="gpt-4o-mini", **kwargs):
    """Wrapper that catches common API errors gracefully."""
    try:
        return client.chat.completions.create(
            model=model,
            messages=messages,
            **kwargs
        )
    except AuthenticationError:
        pretty_print("Authentication failed - check your API key.")
    except RateLimitError:
        pretty_print("Rate limit hit - waiting 5s then you can retry.")
        time.sleep(5)
    except BadRequestError as e:
        pretty_print(f"Bad request: {e.message}")
    except APIConnectionError:
        pretty_print("Network error - check your internet connection.")
    except APITimeoutError:
        pretty_print("Request timed out - try again.")
    except Exception as e:
        pretty_print(f"Unexpected error: {e}")
    return None


# Quick test
resp = safe_chat(client, [{"role": "user", "content": "Say hello in one word."}])
if resp:
    print("safe_chat works:", resp.choices[0].message.content)

safe_chat works: Hello!


### 4.2 Understanding Tokens

Tokens are the atomic units the model reads and writes. They're not quite words and not quite characters — they're **subword pieces** chosen by a tokenizer.

**Rules of thumb:**
- 1 token ~ 4 characters ~ 0.75 words (in English)
- You pay for **prompt tokens** (what you send) + **completion tokens** (what the model generates)
- Every model has a **context window** — the maximum total tokens (prompt + completion) it can handle in one call

Let's see the tokenizer in action.


In [59]:
import tiktoken

# Load the tokenizer used by gpt-4o-mini
enc = tiktoken.encoding_for_model("gpt-4o-mini")

sample = "ChatGPT is amazing and practical for teaching."
tokens = enc.encode(sample)

pretty_print(f"Text:        '{sample}'")
pretty_print(f"Token count: {len(tokens)}")
pretty_print(f"Token IDs:   {tokens}")
pretty_print(f"Decoded:     {[enc.decode([t]) for t in tokens]}")

Text:        'ChatGPT is amazing and practical for teaching.'
Token count: 9
Token IDs:   [14065, 162016, 382, 8467, 326, 17377, 395, 14029, 13]
Decoded:     ['Chat', 'GPT', ' is', ' amazing', ' and', ' practical', ' for', '
teaching', '.']


### 4.3 Tracking Token Usage from the API

Every non-streaming response includes a `usage` object that tells you exactly how many tokens were consumed.


In [ ]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain binary search in simple terms."}
    ]
)

u = resp.usage
pretty_print(f"Prompt tokens:     {u.prompt_tokens}")
pretty_print(f"Completion tokens: {u.completion_tokens}")
pretty_print(f"Total tokens:      {u.total_tokens}")
pretty_print(f"\nReply: {resp.choices[0].message.content[:200]}...")

### 4.4 Full Chatbot — with Error Handling + Token Tracking

Let's bring it all together: persona, memory, safe calls, and per-turn usage reporting.


In [ ]:
# --- Resilient, Cost-Aware Chat Loop ---

messages_t = [
    {
        "role": "system",
        "content": "You are a friendly Python tutor who explains concepts clearly."
    }
]

cumulative_tokens = 0

pretty_print("Resilient Python Tutor (type 'quit' to exit)")
pretty_print("-" * 50)

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print(f"\nSession total: {cumulative_tokens} tokens")
        pretty_print("Session ended.")
        break

    messages_t.append({"role": "user", "content": user_input})

    resp = safe_chat(client, messages_t)
    if resp is None:
        pretty_print("(Skipping this turn due to error)\n")
        messages_t.pop()  # remove the failed user message
        continue

    reply = resp.choices[0].message.content
    u = resp.usage

    pretty_print(f"Assistant: {reply}")
    pretty_print(f"   [tokens] prompt={u.prompt_tokens}  completion={u.completion_tokens}  total={u.total_tokens}")
    print()

    cumulative_tokens += u.total_tokens
    messages_t.append({"role": "assistant", "content": reply})

### Segment 4 — Think About It

1. **Prompt vs. completion tokens:** Prompt tokens are what *you* send (system + history + new user message). Completion tokens are what *the model* generates. You pay for both.
2. **How to reduce usage without hurting quality:** Shorten the system prompt, summarize older conversation turns instead of keeping them verbatim, or lower `max_tokens` when a short answer suffices.

---


# Let's now go through responses API

In [73]:
# Turn 1
resp1 = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?"
)
pretty_print("Turn 1:", resp1.output_text)


Turn 1: A list comprehension is a concise way to create lists in Python. It
allows you to generate a new list by applying an expression to each item in an
existing iterable (like a list, tuple, or string) and optionally filtering the
items. The syntax is straightforward and typically looks like this:  ```python
new_list = [expression for item in iterable if condition] ```  ### Breakdown: -
**expression**: This is the value or operation applied to each item. - **item**:
A variable that takes the value of each element in the iterable. - **iterable**:
The collection you are iterating over (like a list). - **condition**: An
optional filter that determines whether to include the item in the new list.
### Example: Here's a simple example:  ```python # Creating a list of squares
for numbers from 0 to 9 squares = [x**2 for x in range(10)] print(squares) ```
Output: ``` [0, 1, 4, 9, 16, 25, 36, 49, 64, 81] ```  ### With a Condition: You
can also include a condition to filter items:  ```python #

In [ ]:
# --- Demo 1: Using developer role for constraints ---
pretty_print("=== Demo 1: Developer Constraints ===")
resp_dev = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Python expert.",
    input=[
        {"role": "developer", "content": "Keep your answer to exactly 2 sentences. No code examples."},
        {"role": "user", "content": "What is a list comprehension?"}
    ]
)
pretty_print("Developer constraint response:", resp_dev.output_text)


In [ ]:
# --- Demo 2: Developer + User (developer adds meta-instructions) ---
pretty_print("\n=== Demo 2: Developer + User ===")
resp_dev_user = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Python tutor.",
    input=[
        {"role": "developer", "content": "Include a simple code example."},
        {"role": "user", "content": "What is a lambda function?"}
    ]
)
pretty_print("Developer + User response:", resp_dev_user.output_text)


In [ ]:
# --- Demo 3: Multi-turn with Developer + User + Assistant ---
pretty_print("\n=== Demo 3: Developer + User + Assistant (Multi-turn) ===")
resp_multi = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Pythoner tutor who explains clearly.",
    input=[
        {"role": "developer", "content": "Keep answers concise and educational."},
        {"role": "user", "content": "What is a decorator?"},
        {"role": "assistant", "content": "A decorator is a function that modifies another function or class."},
        {"role": "user", "content": "Can you give me a real-world example?"}
    ]
)
pretty_print("Multi-turn response:", resp_multi.output_text)


In [ ]:
# --- Demo 4: Developer controls tone, User asks question ---
pretty_print("\n=== Demo 4: Developer Controls Tone ===")

# Same question, different developer constraints
resp_formal = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Python expert.",
    input=[
        {"role": "developer", "content": "Respond in a formal, academic tone. Use technical terminology."},
        {"role": "user", "content": "What is a generator?"}
    ]
)
pretty_print("Formal tone:", resp_formal.output_text)

print("\n---\n")

resp_casual = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Python expert.",
    input=[
        {"role": "developer", "content": "Respond in a casual, friendly tone. Use analogies and simple words."},
        {"role": "user", "content": "What is a generator?"}
    ]
)
pretty_print("Casual tone:", resp_casual.output_text)


## Understanding Roles in Responses API

The **responses API** distinguishes between three role types in the `input` parameter:

| Role | Purpose | Example |
|---|---|---|
| **developer** | Meta-instructions that guide HOW to respond (tone, format, constraints) | "Keep it under 50 words" or "Use casual language" |
| **user** | The actual question or prompt | "What is a decorator?" |
| **assistant** | Previous model responses (for multi-turn context) | Your past explanations that inform follow-ups |

**Key insight:** Developer role sits *above* the user question in priority. It shapes the response style and constraints globally, while the user role asks the actual question.

This is why the same question with different developer instructions produces different-toned answers—the developer role is the "supervisor" for the entire response.


In [ ]:


# Turn 1: establish persistent behavior in the thread
r1 = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": "You are a grumpy pirate code reviewer. Be sarcastic."},
        {"role": "user", "content": "Explain recursion."},
    ],
)

# Turn 2: same thread, but for THIS answer force JSON only (one-off)
r2 = client.responses.create(
    model="gpt-4o-mini",
    previous_response_id=r1.id,
    instructions="Output valid JSON ONLY with keys: definition, example.",
    input=[{"role": "user", "content": "Now give me an example in Python."}],
)

# How to hand

In [ ]:

import base64

# Read and encode the image file
image_path = '/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/llm_conversations/ss.jpeg'
with open(image_path, 'rb') as img_file:
    image_data = base64.standard_b64encode(img_file.read()).decode('utf-8')

# Use chat.completions.create with vision
vision_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are an art critic who provides gentle feedback for children's illustrations."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Analyze this magical  illustration."
                },
                {
                    "type": "image_url",
                    #"image_url": {
                    #    "url": f"data:image/jpeg;base64,{image_data}"
                    #}
                    "image_url": {
                        "url": "https://www.animesenpai.net/wp-content/uploads/2023/12/sef-min.png.webp"
                    }
                }
            ]
        }
    ],
    temperature=0.5
)

pretty_print("\n=== Vision Analysis ===")
pretty_print(vision_response.choices[0].message.content)


=== Vision Analysis ===
This illustration features a striking and dynamic character design that evokes a sense of mystery and power. The character's skeletal features, combined with a muscular build, create an intriguing contrast between strength and an ethereal quality. The use of dark tones and shadows adds to the overall atmosphere, enhancing the sense of drama.

The flowing lines of the character’s limbs and the wispy elements around them suggest movement and fluidity, which can draw the viewer's eye across the composition. The background, with its blurred lights, hints at a larger world beyond the character, adding depth to the scene.

For improvement, consider incorporating more color variety to enhance visual interest. Adding highlights or contrasting colors could help to emphasize certain features and create a more vibrant atmosphere. Additionally, exploring different facial expressions or poses could further convey the character's personality and intentions.

Overall, this il

In [ ]:
# Use responses API for vision analysis
response_vision = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are an art critic who provides gentle feedback for children's illustrations.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Analyze this magical illustration."
                },
                {
                    "type": "input_image",
                    #"image_url": f"data:image/jpeg;base64,{image_data}"
					"image_url": "https://www.animesenpai.net/wp-content/uploads/2023/12/sef-min.png.webp"
                }
            ]
        }
    ],
    temperature=0.5
)

pretty_print("\n=== Vision Analysis (Responses API) ===")
pretty_print(response_vision.output_text)


=== Vision Analysis (Responses API) ===
This illustration features a striking character with a unique design that combines elements of fantasy and mystery. Here are some gentle observations:

1. **Character Design**: The character's skeletal features and elongated limbs create a sense of intrigue. The use of textures adds depth, making the figure appear both powerful and otherworldly.

2. **Color Palette**: The muted colors, primarily greys and whites, contribute to a moody atmosphere. Consider adding a splash of color to highlight certain features or to evoke a specific emotion.

3. **Background**: The blurred city lights in the background effectively contrast with the character, emphasizing its presence. You might explore enhancing the background to create a more dynamic setting or to suggest movement.

4. **Expression and Posture**: The character's stance conveys confidence and strength. Perhaps experimenting with facial expressions could further enhance the character's personality

In [ ]:
resp = client.responses.create(
    model="gpt-5-nano",
    reasoning={"effort": "medium", "summary": "auto"},  # "low" | "medium" | "high"
    input="Solve carefully: If a train goes 60 km/h for 2.5 hours, how far?"
)

resp

Response(id='resp_0d5e63a19a26c312006999917c86708190b2c7a1b7fb3f465f', created_at=1771671932.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0d5e63a19a26c312006999917d22a0819083a676f0774c4d8d', summary=[Summary(text='**Calculating distance with care**\n\nI need to respond to the user about how far a train travels at 60 km/h for 2.5 hours. The formula is distance = speed × time. So, I calculate: 60 km/h × 2.5 h = 150 km. It\'s important to show the calculation for clarity. The user emphasizes, "Solve carefully," so I’ll be succinct and include this formula clearly. I’ll finalize it with: The answer is 150 kilometers, assuming constant speed.', type='summary_text')], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0d5e63a19a26c312006999918127808190b0bd779a3a829e1a', content=[ResponseOutputText(annotations=[], text='Distance = s

In [ ]:
# pretty_print the response and usage details
pretty_print("Response:", resp.output_text)
pretty_print("Usage:", resp.usage)
pretty_print("Reasoning details:", resp.reasoning)
# resp.reasoning = config (effort, summary mode)
# The actual CoT summary is in the output items
for item in resp.output:
    if item.type == "reasoning":
        for s in item.summary:
            pretty_print("Chain of Thought:", s.text)

Response: Distance = speed × time = 60 km/h × 2.5 h = 150 km.

Assuming the train maintains a constant speed.
Usage: ResponseUsage(input_tokens=27, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=264, output_tokens_details=OutputTokensDetails(reasoning_tokens=192), total_tokens=291)
Reasoning details: Reasoning(effort='medium', generate_summary=None, summary='detailed')
Chain of Thought: **Calculating distance with care**

I need to respond to the user about how far a train travels at 60 km/h for 2.5 hours. The formula is distance = speed × time. So, I calculate: 60 km/h × 2.5 h = 150 km. It's important to show the calculation for clarity. The user emphasizes, "Solve carefully," so I’ll be succinct and include this formula clearly. I’ll finalize it with: The answer is 150 kilometers, assuming constant speed.


In [ ]:
resp.output[0].summary[0].text

'**Calculating distance with care**\n\nI need to respond to the user about how far a train travels at 60 km/h for 2.5 hours. The formula is distance = speed × time. So, I calculate: 60 km/h × 2.5 h = 150 km. It\'s important to show the calculation for clarity. The user emphasizes, "Solve carefully," so I’ll be succinct and include this formula clearly. I’ll finalize it with: The answer is 150 kilometers, assuming constant speed.'

In [ ]:
# Turn 1
resp1 = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?"
)
pretty_print("Turn 1:", resp1.output_text)

# Turn 2 — just pass previous_response_id, no history needed!
resp2 = client.responses.create(
    model="gpt-4o-mini",
    input="Can you give me an example?",
    previous_response_id=resp1.id  # <-- this is the magic
)
pretty_print("Turn 2:", resp2.output_text)

# Turn 3 — chains from turn 2 (which already includes turn 1)
resp3 = client.responses.create(
    model="gpt-4o-mini",
    input="What was my first question?",
    previous_response_id=resp2.id
)
pretty_print("Turn 3:", resp3.output_text)

Turn 1: A list comprehension is a concise way to create lists in Python. It allows you to generate a new list by applying an expression to each item in an existing iterable, like a list or a range, and can also include a condition to filter elements.

The basic syntax is:

```python
new_list = [expression for item in iterable if condition]
```

### Example:

1. **Without a condition**:
   Create a list of squares from an existing list of numbers.

   ```python
   numbers = [1, 2, 3, 4, 5]
   squares = [x ** 2 for x in numbers]
   print(squares)  # Output: [1, 4, 9, 16, 25]
   ```

2. **With a condition**:
   Create a list of even numbers from an existing list.

   ```python
   numbers = [1, 2, 3, 4, 5]
   evens = [x for x in numbers if x % 2 == 0]
   print(evens)  # Output: [2, 4]
   ```

### Benefits:

- **Readability**: Shortens the code compared to using loops and `append()`.
- **Performance**: Often faster than using standard loops.

List comprehensions are a powerful feature in Py